In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
segment_names = ['PB2', 'PB1', 'PA', 'HA', 'NP', 'NA', 'MP', 'NS']

folder = Path("/data/users/ltucker/influenzaData/H5N1_pipeline/output/famsa_distances")
files = sorted(folder.glob("symmetric_distances_*.parquet"))

all_segments = []
for index, file in enumerate(files):
    df = pd.read_parquet(file)
    all_segments.append(df)
    print(f"{segment_names[index]}: {df.shape}")

In [ ]:
gmm_results = []

for index, df in enumerate(all_segments):
    upper_tri = df.values[np.triu_indices_from(df.values, k=1)].reshape(-1, 1)

    # Select best k by BIC
    best_bic = np.inf
    best_gmm = None
    best_k = 2
    for k in range(2, 7):
        gmm = GaussianMixture(n_components=k, random_state=42).fit(upper_tri)
        bic = gmm.bic(upper_tri)
        if bic < best_bic:
            best_bic = bic
            best_gmm = gmm
            best_k = k

    gmm_results.append({"segment": segment_names[index], "k": best_k, "gmm": best_gmm})
    print(f"{segment_names[index]}: best k={best_k}, BIC={best_bic:.0f}")

In [ ]:
fig, axes = plt.subplots(len(all_segments), 1, figsize=(12, 4 * len(all_segments)))

colors = plt.cm.tab10.colors

for index, df in enumerate(all_segments):
    ax = axes[index]
    upper_tri = df.values[np.triu_indices_from(df.values, k=1)]
    gmm = gmm_results[index]["gmm"]
    k = gmm_results[index]["k"]

    # Histogram (density-normalized so it matches the PDF curves)
    ax.hist(upper_tri, bins=150, density=True, alpha=0.3, color="gray", edgecolor="none")

    # Plot each component
    x = np.linspace(upper_tri.min(), upper_tri.max(), 1000)
    total_pdf = np.zeros_like(x)

    for j in range(k):
        weight = gmm.weights_[j]
        mean = gmm.means_[j, 0]
        std = np.sqrt(gmm.covariances_[j, 0, 0])
        component_pdf = weight * norm.pdf(x, mean, std)
        total_pdf += component_pdf
        ax.plot(x, component_pdf, color=colors[j % len(colors)], linewidth=1.5,
                label=f"k{j+1}: μ={mean:.3f}, σ={std:.3f}, w={weight:.2f}")

    # Combined fit
    ax.plot(x, total_pdf, color="black", linewidth=2, linestyle="--", label="Combined fit")
    ax.set_title(f"{segment_names[index]} (k={k})")
    ax.set_xlabel("Distance")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()